Aim of this script: identify the rail operator for each row, using the `data_chuuchuu_{data_selection}_terminus.parquet` intermediate output produced by `Chuuchuu_data_test_terminus_identification.ipynb`.

In [1]:
import pandas as pd
import numpy as np
import os

In [2]:
data_selection = "french"

intermediate_outputs_dir = "intermediate_outputs"
data_path = f"{intermediate_outputs_dir}/data_chuuchuu_{data_selection}_terminus.parquet"

try: 
    data_chuuchuu.head()
except NameError:
    data_chuuchuu = pd.read_parquet(data_path)
data_chuuchuu.shape

(2894465, 35)

In [3]:
# some data_selection subsets (e.g. "french") have no operator info at all, so pandas infers
# an all-NaN float64 dtype for this column on load -- cast to object so the string assignments
# below don't fail with a dtype mismatch
data_chuuchuu["operator"] = data_chuuchuu["operator"].astype("object")

### Helper columns & constants

Translating the SQL rules from `instructions_chuuchuu_operator_identification.md` into pandas:

- `UPPER(TRIM(col))` comparisons -> normalized `_norm` columns.
- `routeNumber` list/CAST(...AS INTEGER) BETWEEN comparisons -> a trimmed string column plus a numeric column (non-numeric values become `NaN` and simply won't match any range check).
- `stopcountry = 'XX'` -> the full country name already resolved in this dataset's `country` column (see `Chuuchuu_data_test_country_attribution.ipynb`).

Rules are applied in the same order as the instructions doc, so later rules intentionally overwrite `operator` for rows also matched by an earlier, broader rule -- exactly like running the SQL `UPDATE`s in sequence.

In [4]:
def norm(series):
    """Mirror SQL's UPPER(TRIM(...)) comparison."""
    return series.astype(str).str.strip().str.upper()

def in_ranges(int_series, ranges):
    """OR together several inclusive (lo, hi) BETWEEN checks; NaN (non-numeric routeNumber) never matches."""
    mask = pd.Series(False, index=int_series.index)
    for lo, hi in ranges:
        mask |= int_series.between(lo, hi)
    return mask

data_chuuchuu["routeType_norm"] = norm(data_chuuchuu["routeType"])
data_chuuchuu["agency_norm"] = norm(data_chuuchuu["agency"])
data_chuuchuu["routeNumber_trim"] = data_chuuchuu["routeNumber"].astype(str).str.strip()
data_chuuchuu["routeNumber_int"] = pd.to_numeric(data_chuuchuu["routeNumber_trim"], errors="coerce")

print(f"{data_chuuchuu['routeNumber_int'].isna().sum()} rows have a non-numeric routeNumber (can't be matched by any routeNumber-range rule below)")

0 rows have a non-numeric routeNumber (can't be matched by any routeNumber-range rule below)


### Highspeed services

In [5]:
# routeType matched with its exact (case-sensitive) literal, same as the source SQL
highspeed_operator_by_routeType = {
    "ICE": "Deutsche Bahn",
    "TGV": "SNCF", "TGV INOUI": "SNCF", "TGV Lyria": "SNCF", "LYR": "SNCF", "LYRIA": "SNCF", "Ouigo": "SNCF", "OUI": "SNCF",
    "RJX": "OEBB", "rjx": "OEBB",
    "FR": "Trenitalia",
    "EST": "Eurostar", "EUR": "Eurostar", "Eurostar": "Eurostar",
    "Italo": "Italo",
}

highspeed_mask = data_chuuchuu["routeType"].isin(highspeed_operator_by_routeType)
data_chuuchuu.loc[highspeed_mask, "operator"] = data_chuuchuu.loc[highspeed_mask, "routeType"].map(highspeed_operator_by_routeType)
print(f"{highspeed_mask.sum()} rows assigned an operator via the highspeed routeType rule")

995070 rows assigned an operator via the highspeed routeType rule


### Night train services

`NJ`/`Nightjet` -> OEBB, `ES`/`European Sleeper` -> European Sleeper, and `EN` split by `routeNumber` (a handful of `EN` route numbers are first relabeled to `NJ`, per the instructions).

In [6]:
# relabel the NJ-under-EN route numbers first, so the NJ operator rule below picks them up too
en_to_nj_route_numbers = {"294", "295", "13485", "233"}
en_to_nj_mask = (data_chuuchuu["routeType_norm"] == "EN") & data_chuuchuu["routeNumber_trim"].isin(en_to_nj_route_numbers)
data_chuuchuu.loc[en_to_nj_mask, "routeType"] = "NJ"
data_chuuchuu.loc[en_to_nj_mask, "routeType_norm"] = "NJ"
print(f"{en_to_nj_mask.sum()} rows had routeType 'EN' relabeled to 'NJ'")

data_chuuchuu.loc[data_chuuchuu["routeType_norm"].isin(["ES", "EUROPEAN SLEEPER"]), "operator"] = "European Sleeper"
data_chuuchuu.loc[data_chuuchuu["routeType_norm"].isin(["NJ", "NIGHTJET"]), "operator"] = "OEBB"

en_operator_by_routeNumber = {
    "SJ": {"344", "345", "346", "13471", "13472"},
    "HZ": {"40465", "40414", "40237", "414", "415"},
    "PKP": {"406", "407", "40417", "40416", "40407", "1276", "1277"},
    "Ceske Drahy": {"40458", "40459", "443", "442"},
    "MAV": {"40462", "40467", "50237", "50462", "40476", "40457", "40406", "462", "476", "477", "463"},
}
for operator_name, route_numbers in en_operator_by_routeNumber.items():
    mask = (data_chuuchuu["routeType_norm"] == "EN") & data_chuuchuu["routeNumber_trim"].isin(route_numbers)
    data_chuuchuu.loc[mask, "operator"] = operator_name
    print(f"{mask.sum()} EN rows assigned operator '{operator_name}'")

# route numbers the instructions explicitly flag as having no known operator yet -- left untouched
en_unknown_route_numbers = {
    "1415", "1153", "1152", "50476", "323",
    "13400", "13403", "13451", "13408", "13401", "13402", "13404", "13406", "13420", "13405", "13409", "13417",
    "93701", "319", "580", "230", "320", "34834", "91641", "91505", "32",
    "277", "37501", "89843", "89962", "11799", "28565", "28960", "370", "20159",
}
en_unknown_mask = (data_chuuchuu["routeType_norm"] == "EN") & data_chuuchuu["routeNumber_trim"].isin(en_unknown_route_numbers)
print(f"{en_unknown_mask.sum()} EN rows have no known operator per the instructions -- left as-is")

0 rows had routeType 'EN' relabeled to 'NJ'


0 EN rows assigned operator 'SJ'
0 EN rows assigned operator 'HZ'
0 EN rows assigned operator 'PKP'


0 EN rows assigned operator 'Ceske Drahy'
0 EN rows assigned operator 'MAV'


0 EN rows have no known operator per the instructions -- left as-is


### Railjet (ÖBB or Ceske Drahy)

In [7]:
ceske_drahy_rj_route_numbers = {
    "50", "51", "52", "53", "54", "55", "56",
    "70", "71", "72", "73", "74", "75", "78", "79",
    "170", "171", "172", "173", "174", "175", "176", "177", "178", "179",
    "244", "250", "251", "252", "253", "254", "255", "256", "257", "258", "259",
    "270", "271", "272", "273", "274", "275", "276", "277",
    "284", "285",
    "370", "371", "372", "373", "374", "375",
    "382", "383", "384", "385", "386", "387",
    "478", "479",
    "512", "515",
    "548", "549",
    "576", "577", "578", "579",
    "644", "645",
}

rj_mask = data_chuuchuu["routeType_norm"] == "RJ"
is_ceske_drahy_rj = rj_mask & data_chuuchuu["routeNumber_trim"].isin(ceske_drahy_rj_route_numbers)
data_chuuchuu.loc[is_ceske_drahy_rj, "operator"] = "Ceske Drahy"
data_chuuchuu.loc[rj_mask & ~is_ceske_drahy_rj, "operator"] = "OEBB"
print(f"{is_ceske_drahy_rj.sum()} RJ rows assigned 'Ceske Drahy', {(rj_mask & ~is_ceske_drahy_rj).sum()} assigned 'OEBB'")

0 RJ rows assigned 'Ceske Drahy', 0 assigned 'OEBB'


### Easy operator cases (PL, FLX, HU, IT, FR)

In [8]:
pl_mask = data_chuuchuu["agency"] == "PL"
data_chuuchuu.loc[pl_mask, "operator"] = "PKP Intercity"

flx_mask = data_chuuchuu["routeType_norm"] == "FLX"
data_chuuchuu.loc[flx_mask, "operator"] = "Flixtrain"

hu_routeTypes = {"EC", "IC", "EX", "EN", "G", "GY", "ER", "H", "IR", "S", "SZ", "Z"}
hu_mask = (data_chuuchuu["agency_norm"] == "HU") & data_chuuchuu["routeType_norm"].isin(hu_routeTypes)
data_chuuchuu.loc[hu_mask, "operator"] = "MAV"

it_routeTypes = {"FR", "FA", "FB", "IC", "ICN", "EXP", "IR", "REG", "MET", "EN", "NCL"}
it_mask = (data_chuuchuu["agency_norm"] == "IT") & data_chuuchuu["routeType_norm"].isin(it_routeTypes)
data_chuuchuu.loc[it_mask, "operator"] = "Trenitalia"

fr_routeTypes = {
    "IC", "ICN", "INTERCITES", "INTERCITES DE NUIT", "LYR", "LYRIA",
    "NAV", "NAVETTE", "OGO", "OUI", "OUIGO", "TER", "TGV INOUI", "TRAIN TER",
}
fr_mask = (data_chuuchuu["agency_norm"] == "FR") & data_chuuchuu["routeType_norm"].isin(fr_routeTypes)
data_chuuchuu.loc[fr_mask, "operator"] = "SNCF"

print(f"PL: {pl_mask.sum()}, FLX: {flx_mask.sum()}, HU: {hu_mask.sum()}, IT: {it_mask.sum()}, FR: {fr_mask.sum()}")

PL: 0, FLX: 0, HU: 0, IT: 0, FR: 2517370


### SBB agency

`IC`/`EC` -> SBB whenever the stop is in Switzerland. `IR`/`R`/`RE`/`S`/`SN` are only *sometimes* SBB, split by `routeNumber` blocks. `PE`/`ICE`/`RB` have no rule.

In [9]:
# IC, EC inside Switzerland -> SBB
sbb_ic_ec_mask = data_chuuchuu["routeType"].isin(["IC", "EC"]) & (data_chuuchuu["country"] == "Switzerland")
data_chuuchuu.loc[sbb_ic_ec_mask, "operator"] = "SBB"
print(f"{sbb_ic_ec_mask.sum()} IC/EC rows in Switzerland assigned 'SBB'")

# IR
sbb_ir_route_numbers = {
    "1651", "1652", "1653", "1654", "1656", "1658", "1662", "1671", "1673", "1674", "1675", "1676", "1677", "1678",
    "1900", "1902", "1904", "1910", "1929", "1943", "1945",
    "2306", "2344", "2354", "2357", "2358", "2361", "2366", "2370", "2374", "2379", "2381", "2388", "2390", "2392", "2393", "2394",
    "3016", "3017", "3022", "3024", "3026", "3029", "3030", "3110",
    "746", "749",
}
sbb_ir_ranges = [(1702, 1843), (1956, 1995), (2055, 2194), (2252, 2292), (2456, 2493), (2503, 2543), (2562, 2599), (2610, 2662), (3251, 3293)]

sbb_ir_mask = (
    (data_chuuchuu["agency"] == "SBB") & (data_chuuchuu["routeType"] == "IR")
    & (data_chuuchuu["routeNumber_trim"].isin(sbb_ir_route_numbers) | in_ranges(data_chuuchuu["routeNumber_int"], sbb_ir_ranges))
)
data_chuuchuu.loc[sbb_ir_mask, "operator"] = "SBB"
print(f"{sbb_ir_mask.sum()} SBB IR rows assigned 'SBB'")

0 IC/EC rows in Switzerland assigned 'SBB'
0 SBB IR rows assigned 'SBB'


In [10]:
# R
sbb_r_ranges = [
    (14402, 14693), (17400, 17497), (18940, 18999), (23000, 23535), (24004, 24999),
    (25850, 25864), (26101, 26370), (5607, 5994), (6006, 6758), (7014, 7461), (9901, 9940),
]
sbb_r_mask = (data_chuuchuu["agency"] == "SBB") & (data_chuuchuu["routeType"] == "R") & in_ranges(data_chuuchuu["routeNumber_int"], sbb_r_ranges)
data_chuuchuu.loc[sbb_r_mask, "operator"] = "SBB"
print(f"{sbb_r_mask.sum()} SBB R rows assigned 'SBB'")

# RE
sbb_re_route_numbers = {"2087", "2089", "2090", "2092", "2560", "741", "742", "744"}
sbb_re_ranges = [(18122, 18495), (25500, 25843), (2600, 2607), (3564, 3685), (3958, 3995), (4706, 4716), (4762, 4842), (4908, 4940)]
sbb_re_mask = (
    (data_chuuchuu["agency"] == "SBB") & (data_chuuchuu["routeType"] == "RE")
    & (data_chuuchuu["routeNumber_trim"].isin(sbb_re_route_numbers) | in_ranges(data_chuuchuu["routeNumber_int"], sbb_re_ranges))
)
data_chuuchuu.loc[sbb_re_mask, "operator"] = "SBB"
print(f"{sbb_re_mask.sum()} SBB RE rows assigned 'SBB'")

0 SBB R rows assigned 'SBB'
0 SBB RE rows assigned 'SBB'


In [11]:
# S
sbb_s_route_numbers = {
    "30392", "30661", "30698", "30794", "30821", "30823", "30825", "30827", "30829", "30831", "30833", "30835", "30837", "30839",
    "30841", "30843", "30845", "30847", "30849", "30851", "30853", "30855", "30857", "30859", "30861", "30863", "30865", "30867",
    "30869", "30871", "30873", "30949", "30967", "30985",
}
sbb_s_ranges = [
    (14303, 14398),
    (17001, 17099), (17113, 17394), (17525, 17568), (17906, 17943),
    (18011, 18020), (18087, 18093), (18220, 18993),
    (19010, 19293), (19416, 19693), (19921, 19980),
    (20024, 20580),
    (21016, 21395), (21912, 21997),
    (22015, 22175), (22960, 22991),
    (24500, 24697),
    (25100, 25792), (25905, 25993),
    (26018, 26075),
    (7606, 7743), (7819, 7892),
    (8416, 8999),
]
sbb_s_mask = (
    (data_chuuchuu["agency"] == "SBB") & (data_chuuchuu["routeType"] == "S")
    & (data_chuuchuu["routeNumber_trim"].isin(sbb_s_route_numbers) | in_ranges(data_chuuchuu["routeNumber_int"], sbb_s_ranges))
)
data_chuuchuu.loc[sbb_s_mask, "operator"] = "SBB"
print(f"{sbb_s_mask.sum()} SBB S rows assigned 'SBB'")

# SN
sbb_sn_route_numbers = {
    "13702", "13704", "13707", "13709", "13710", "13711", "13712", "13713", "13714", "13715",
    "13716", "13717", "13746", "13748", "13750", "13751", "13752", "13753", "13754", "13755",
    "13756", "13757", "13760", "13762", "13763", "13764", "13765", "13766", "13767", "13769",
    "13770", "13771", "13772", "13773", "13774", "13775", "13776", "13777", "13780", "13781",
    "13782", "13783", "13784", "13785", "13786", "13787", "13790", "13791", "13792", "13793",
    "13794", "13795", "13796", "13797", "13811", "13812", "13813", "13814", "13815", "13816",
    "13818", "13830", "13831", "13832", "13834", "13835", "13836", "13837", "13838", "13839",
    "13840", "13841", "13842", "13843", "13844", "13845", "13846", "13847", "13849",
    "87795", "87797",
}
sbb_sn_mask = (data_chuuchuu["agency"] == "SBB") & (data_chuuchuu["routeType"] == "SN") & data_chuuchuu["routeNumber_trim"].isin(sbb_sn_route_numbers)
data_chuuchuu.loc[sbb_sn_mask, "operator"] = "SBB"
print(f"{sbb_sn_mask.sum()} SBB SN rows assigned 'SBB'")

print("SBB PE / ICE / RB: no operator rule provided -- left as-is")

0 SBB S rows assigned 'SBB'
0 SBB SN rows assigned 'SBB'
SBB PE / ICE / RB: no operator rule provided -- left as-is


### OEBB agency

`CJX`/`D`/`ER`/`IR` are always OEBB in Austria. `EC`/`IC` split by `routeNumber` with an OEBB catch-all. `Os` is treated as exclusively Ceske Drahy. `R`/`REX`/`S` are OEBB in Austria except for a few excluded route numbers/ranges (the ones run by other, smaller operators per the observed counts). `RB`/`UEX` have no rule. `WB` is always Westbahn.

In [12]:
oebb_always_mask = data_chuuchuu["routeType"].isin(["CJX", "D", "ER", "IR"]) & (data_chuuchuu["country"] == "Austria")
data_chuuchuu.loc[oebb_always_mask, "operator"] = "OEBB"
print(f"{oebb_always_mask.sum()} CJX/D/ER/IR rows in Austria assigned 'OEBB'")

# EC
oebb_ec_base = (data_chuuchuu["country"] == "Austria") & (data_chuuchuu["routeType"] == "EC")

oebb_ec_oebb_numbers = {
    "100", "102", "106", "114", "140", "141", "142", "143", "144", "145", "146", "147", "148", "149", "164",
    "202", "204", "206", "212", "214", "290", "337", "340", "341", "342", "343", "462", "463", "70", "71", "78", "79",
}
oebb_ec_db_numbers = {"80", "81", "94", "96", "98", "115", "190", "192", "194", "196", "198", "213", "1281"}
oebb_ec_sbb_numbers = {"95", "97", "99", "163", "191", "193", "195", "197", "199"}
oebb_ec_pkp_numbers = {"101", "103", "107", "203", "205", "207"}

data_chuuchuu.loc[oebb_ec_base & data_chuuchuu["routeNumber_trim"].isin(oebb_ec_oebb_numbers), "operator"] = "OEBB"
data_chuuchuu.loc[oebb_ec_base & data_chuuchuu["routeNumber_trim"].isin(oebb_ec_db_numbers), "operator"] = "DB"
data_chuuchuu.loc[oebb_ec_base & data_chuuchuu["routeNumber_trim"].isin(oebb_ec_sbb_numbers), "operator"] = "SBB"
data_chuuchuu.loc[oebb_ec_base & data_chuuchuu["routeNumber_trim"].isin(oebb_ec_pkp_numbers), "operator"] = "PKP Intercity"

oebb_ec_listed_numbers = oebb_ec_oebb_numbers | oebb_ec_db_numbers | oebb_ec_sbb_numbers | oebb_ec_pkp_numbers
oebb_ec_catchall = oebb_ec_base & ~data_chuuchuu["routeNumber_trim"].isin(oebb_ec_listed_numbers)
data_chuuchuu.loc[oebb_ec_catchall, "operator"] = "OEBB"
print(f"{oebb_ec_base.sum()} OEBB EC rows in Austria processed ({oebb_ec_catchall.sum()} via the OEBB catch-all)")

0 CJX/D/ER/IR rows in Austria assigned 'OEBB'


0 OEBB EC rows in Austria processed (0 via the OEBB catch-all)


In [13]:
# IC
oebb_ic_base = (data_chuuchuu["country"] == "Austria") & (data_chuuchuu["routeType"] == "IC")

oebb_ic_db_numbers = {"406", "416"}
oebb_ic_pkp_numbers = {"207", "417"}

data_chuuchuu.loc[oebb_ic_base & data_chuuchuu["routeNumber_trim"].isin(oebb_ic_db_numbers), "operator"] = "DB"
data_chuuchuu.loc[oebb_ic_base & data_chuuchuu["routeNumber_trim"].isin(oebb_ic_pkp_numbers), "operator"] = "PKP Intercity"

oebb_ic_listed_numbers = oebb_ic_db_numbers | oebb_ic_pkp_numbers
oebb_ic_catchall = oebb_ic_base & ~data_chuuchuu["routeNumber_trim"].isin(oebb_ic_listed_numbers)
data_chuuchuu.loc[oebb_ic_catchall, "operator"] = "OEBB"
print(f"{oebb_ic_base.sum()} OEBB IC rows in Austria processed ({oebb_ic_catchall.sum()} via the OEBB catch-all)")

# Os -- treated as exclusively Ceske Drahy per the instructions (scoped to the OEBB agency, matching where this was observed)
os_mask = (data_chuuchuu["agency"] == "OEBB") & (data_chuuchuu["routeType"] == "Os")
data_chuuchuu.loc[os_mask, "operator"] = "Ceske Drahy"
print(f"{os_mask.sum()} OEBB 'Os' rows assigned 'Ceske Drahy'")

0 OEBB IC rows in Austria processed (0 via the OEBB catch-all)
0 OEBB 'Os' rows assigned 'Ceske Drahy'


In [14]:
# R
oebb_r_excluded_numbers = {"7807", "1826", "1828"}
oebb_r_mask = (
    (data_chuuchuu["country"] == "Austria") & (data_chuuchuu["routeType"] == "R")
    & ~data_chuuchuu["routeNumber_trim"].isin(oebb_r_excluded_numbers)
    & ~data_chuuchuu["routeNumber_int"].between(8000, 8200)
)
data_chuuchuu.loc[oebb_r_mask, "operator"] = "OEBB"
print(f"{oebb_r_mask.sum()} OEBB R rows in Austria assigned 'OEBB'")
print("OEBB RB: no operator rule provided -- left as-is")

# REX
oebb_rex_excluded = (
    (data_chuuchuu["routeNumber_trim"] == "5572")
    | data_chuuchuu["routeNumber_int"].between(8450, 8600)
    | data_chuuchuu["routeNumber_int"].between(7600, 7900)
)
oebb_rex_mask = (data_chuuchuu["country"] == "Austria") & (data_chuuchuu["routeType"] == "REX") & ~oebb_rex_excluded
data_chuuchuu.loc[oebb_rex_mask, "operator"] = "OEBB"
print(f"{oebb_rex_mask.sum()} OEBB REX rows in Austria assigned 'OEBB'")

0 OEBB R rows in Austria assigned 'OEBB'
OEBB RB: no operator rule provided -- left as-is
0 OEBB REX rows in Austria assigned 'OEBB'


In [15]:
# S
oebb_s_excluded_numbers = {
    "5556", "5562", "5564", "5568", "5572", "5574", "5578", "5580", "5584", "5586",
    "5590", "5592", "5596", "5602", "5606", "5694", "25844", "25846",
    "25883", "25885", "25887", "25889", "25891", "25893", "25895", "25897",
}
oebb_s_excluded = (
    data_chuuchuu["routeNumber_int"].between(4350, 4378)
    | data_chuuchuu["routeNumber_int"].between(7350, 7388)
    | data_chuuchuu["routeNumber_int"].between(8000, 8544)
    | data_chuuchuu["routeNumber_trim"].isin(oebb_s_excluded_numbers)
)
oebb_s_mask = (data_chuuchuu["country"] == "Austria") & (data_chuuchuu["routeType"] == "S") & ~oebb_s_excluded
data_chuuchuu.loc[oebb_s_mask, "operator"] = "OEBB"
print(f"{oebb_s_mask.sum()} OEBB S rows in Austria assigned 'OEBB'")
print("OEBB UEX: no operator rule provided -- left as-is")

# WB
wb_mask = data_chuuchuu["routeType"] == "WB"
data_chuuchuu.loc[wb_mask, "operator"] = "Westbahn"
print(f"{wb_mask.sum()} WB rows assigned 'Westbahn'")

0 OEBB S rows in Austria assigned 'OEBB'
OEBB UEX: no operator rule provided -- left as-is
0 WB rows assigned 'Westbahn'


### DB agency

`EC`/`ECE` split by `routeNumber`, each with its own catch-all label (`EC`'s catch-all is `'DB'`, `ECE`'s is the more specific `'DB Fernverkehr AG'` -- kept exactly as given rather than harmonized). `GV` is always Govolta. `IC` is split by `routeNumber` too, with `'DB Fernverkehr AG'` as its catch-all.

In [16]:
# EC
db_ec_base = (data_chuuchuu["agency"] == "DB") & (data_chuuchuu["routeType"] == "EC")

db_ec_pkp_numbers = {
    "231", "247", "249", "41", "43", "431", "45", "47", "49", "55", "57", "59",
    "230", "246", "248", "40", "42", "430", "44", "46", "48", "54", "56", "58",
}
db_ec_sbb_numbers = {"150", "191", "193", "195", "197", "199", "95", "97", "99", "458", "459"}
db_ec_oebb_numbers = {"213", "115", "114", "212", "290"}

data_chuuchuu.loc[db_ec_base & data_chuuchuu["routeNumber_trim"].isin(db_ec_pkp_numbers), "operator"] = "PKP Intercity"
data_chuuchuu.loc[db_ec_base & data_chuuchuu["routeNumber_trim"].isin(db_ec_sbb_numbers), "operator"] = "SBB"
data_chuuchuu.loc[db_ec_base & data_chuuchuu["routeNumber_trim"].isin(db_ec_oebb_numbers), "operator"] = "OEBB"

db_ec_listed_numbers = db_ec_pkp_numbers | db_ec_sbb_numbers | db_ec_oebb_numbers
db_ec_catchall = db_ec_base & ~data_chuuchuu["routeNumber_trim"].isin(db_ec_listed_numbers)
data_chuuchuu.loc[db_ec_catchall, "operator"] = "DB"
print(f"{db_ec_base.sum()} DB EC rows processed ({db_ec_catchall.sum()} via the 'DB' catch-all)")

# ECE
db_ece_base = (data_chuuchuu["agency"] == "DB") & (data_chuuchuu["routeType"] == "ECE")
db_ece_sbb_numbers = {"190", "192", "194", "196", "198", "94", "96", "98", "151"}

data_chuuchuu.loc[db_ece_base & data_chuuchuu["routeNumber_trim"].isin(db_ece_sbb_numbers), "operator"] = "SBB"
db_ece_catchall = db_ece_base & ~data_chuuchuu["routeNumber_trim"].isin(db_ece_sbb_numbers)
data_chuuchuu.loc[db_ece_catchall, "operator"] = "DB Fernverkehr AG"
print(f"{db_ece_base.sum()} DB ECE rows processed ({db_ece_catchall.sum()} via the 'DB Fernverkehr AG' catch-all)")

# GV
gv_mask = data_chuuchuu["routeType"] == "GV"
data_chuuchuu.loc[gv_mask, "operator"] = "Govolta"
print(f"{gv_mask.sum()} GV rows assigned 'Govolta'")

# IC
db_ic_base = (data_chuuchuu["agency"] == "DB") & (data_chuuchuu["routeType"] == "IC")

db_ic_sbb_numbers = {
    "180", "182", "184", "186", "188", "280", "282", "284", "380", "388", "480", "482", "484", "486", "488",
    "1082", "1180", "181", "183", "185", "187", "281", "283", "285", "389", "1089",
}
db_ic_pkp_numbers = {"132", "134", "407", "417"}
db_ic_oebb_numbers = {"406", "416"}
db_ic_dsb_mask = data_chuuchuu["routeNumber_int"].between(5751, 5767)

data_chuuchuu.loc[db_ic_base & data_chuuchuu["routeNumber_trim"].isin(db_ic_sbb_numbers), "operator"] = "SBB"
data_chuuchuu.loc[db_ic_base & data_chuuchuu["routeNumber_trim"].isin(db_ic_pkp_numbers), "operator"] = "PKP Intercity"
data_chuuchuu.loc[db_ic_base & data_chuuchuu["routeNumber_trim"].isin(db_ic_oebb_numbers), "operator"] = "OEBB"
data_chuuchuu.loc[db_ic_base & db_ic_dsb_mask, "operator"] = "DSB"

db_ic_listed_numbers = db_ic_sbb_numbers | db_ic_pkp_numbers | db_ic_oebb_numbers
db_ic_catchall = db_ic_base & ~data_chuuchuu["routeNumber_trim"].isin(db_ic_listed_numbers) & ~db_ic_dsb_mask
data_chuuchuu.loc[db_ic_catchall, "operator"] = "DB Fernverkehr AG"
print(f"{db_ic_base.sum()} DB IC rows processed ({db_ic_catchall.sum()} via the 'DB Fernverkehr AG' catch-all)")

0 DB EC rows processed (0 via the 'DB' catch-all)


0 DB ECE rows processed (0 via the 'DB Fernverkehr AG' catch-all)
0 GV rows assigned 'Govolta'


0 DB IC rows processed (0 via the 'DB Fernverkehr AG' catch-all)


### GTFSDE agency

`EC`/`ECE`/`EN` are intentionally left unassigned here -- they duplicate DB's own reporting of the same trains (already handled in the DB agency section above), so assigning them again would be redundant. `FEX`/`MEX`/`NRB`/`RB`/`RE`/`RS`/`U` are split by `routeNumber` (and, for `RB`/`RE`, also by `deutscheBahnStopId`, since routeNumber ranges alone overlap too much between DB regional entities and Arverio in those two categories).

In [17]:
# FEX
gtfsde_fex_mask = (data_chuuchuu["agency"] == "GTFSDE") & (data_chuuchuu["routeType"] == "FEX") & data_chuuchuu["routeNumber_int"].between(21800, 21969)
data_chuuchuu.loc[gtfsde_fex_mask, "operator"] = "DB"
print(f"{gtfsde_fex_mask.sum()} GTFSDE FEX rows assigned 'DB'")

# MEX
gtfsde_mex_base = (data_chuuchuu["agency"] == "GTFSDE") & (data_chuuchuu["routeType"] == "MEX")
mex_db_ranges = [(17500, 17570), (19200, 19399), (19500, 19699)]
mex_arverio_ranges = [(19100, 19199), (19400, 19499)]

mex_db_mask = gtfsde_mex_base & in_ranges(data_chuuchuu["routeNumber_int"], mex_db_ranges)
mex_arverio_mask = gtfsde_mex_base & in_ranges(data_chuuchuu["routeNumber_int"], mex_arverio_ranges)
data_chuuchuu.loc[mex_db_mask, "operator"] = "DB"
data_chuuchuu.loc[mex_arverio_mask, "operator"] = "Arverio"
print(f"{mex_db_mask.sum()} GTFSDE MEX rows assigned 'DB', {mex_arverio_mask.sum()} assigned 'Arverio'")

# NRB -- all of it is DB RegioNetz
gtfsde_nrb_mask = (data_chuuchuu["agency"] == "GTFSDE") & (data_chuuchuu["routeType"] == "NRB")
data_chuuchuu.loc[gtfsde_nrb_mask, "operator"] = "DB"
print(f"{gtfsde_nrb_mask.sum()} GTFSDE NRB rows assigned 'DB'")

# RB -- routeNumber ranges alone overlap too much between operators, so DB additionally requires
# a specific set of deutscheBahnStopId values (the stops actually served by that DB regional entity)
gtfsde_rb_base = (data_chuuchuu["agency"] == "GTFSDE") & (data_chuuchuu["routeType"] == "RB")

rb_arverio_ranges = [(57000, 57344), (78901, 78975)]
rb_arverio_mask = gtfsde_rb_base & in_ranges(data_chuuchuu["routeNumber_int"], rb_arverio_ranges)
data_chuuchuu.loc[rb_arverio_mask, "operator"] = "Arverio"

rb_db_stop_ids = ['331064', '360304', '5100082', '5100083', '5100096', '5100222', '5101281', '5102886', '5189954', '5193610', '8010016', '8010018', '8010036', '8010041', '8010051', '8010053', '8010066', '8010069', '8010072', '8010073', '8010079', '8010089', '8010093', '8010099', '8010100', '8010103', '8010113', '8010176', '8010183', '8010193', '8010215', '8010255', '8010279', '8010280', '8010285', '8010300', '8010304', '8010308', '8010322', '8010324', '8010327', '8010338', '8010355', '8010357', '8010373', '8010377', '8010381', '8010389', '8010392', '8010395', '8010396', '8010403', '8010404', '8010405', '8010406', '8011031', '8011078', '8011093', '8011098', '8011102', '8011108', '8011109', '8011114', '8011140', '8011155', '8011160', '8011162', '8011167', '8011179', '8011188', '8011201', '8011270', '8011286', '8011306', '8011318', '8011319', '8011320', '8011334', '8011340', '8011414', '8011419', '8011421', '8011425', '8011471', '8011540', '8011542', '8011563', '8011667', '8011695', '8011729', '8011735', '8011749', '8011778', '8011797', '8011889', '8011901', '8011944', '8011945', '8011991', '8011992', '8011995', '8012006', '8012065', '8012084', '8012086', '8012089', '8012096', '8012108', '8012127', '8012169', '8012215', '8012253', '8012315', '8012316', '8012329', '8012341', '8012377', '8012445', '8012469', '8012479', '8012482', '8012503', '8012582', '8012583', '8012584', '8012609', '8012617', '8012621', '8012650', '8012666', '8012681', '8012713', '8012729', '8012785', '8012806', '8012818', '8012819', '8012840', '8012841', '8012892', '8012903', '8012934', '8012941', '8012962', '8012963', '8013021', '8013040', '8013105', '8013106', '8013132', '8013133', '8013160', '8013161', '8013183', '8013185', '8013267', '8013272', '8013305', '8013339', '8013340', '8013341', '8013350', '8013368', '8013385', '8013406', '8013470', '8013475', '8013481', '8013483', '8013487', '8013489', '8013490', '8017349', '8079084', '8079604', '8079629', '8080170', '8080190', '8080260', '8080370', '8080710', '8081220', '8081688', '8087026', '8087027', '936003']
rb_db_ranges = [
    (5383, 5395), (5830, 5841), (18100, 18166), (18238, 18275), (18300, 18341),
    (93250, 93252), (93268, 93270), (94740, 94775), (18700, 18737), (18748, 18749), (18800, 18869),
    (13030, 13037), (18000, 18048), (18080, 18084), (18550, 18599),
    (13100, 13149), (13221, 13269), (18740, 18745), (18870, 18899), (18345, 18372), (18420, 18445),
]
rb_db_single_numbers = {"18451", "18480", "3648", "18086"}

rb_db_mask = (
    gtfsde_rb_base
    & data_chuuchuu["deutscheBahnStopId"].isin(rb_db_stop_ids)
    & (in_ranges(data_chuuchuu["routeNumber_int"], rb_db_ranges) | data_chuuchuu["routeNumber_trim"].isin(rb_db_single_numbers))
)
data_chuuchuu.loc[rb_db_mask, "operator"] = "DB"
print(f"{rb_arverio_mask.sum()} GTFSDE RB rows assigned 'Arverio', {rb_db_mask.sum()} assigned 'DB'")

# RE -- same stopId + routeNumber approach as RB
gtfsde_re_base = (data_chuuchuu["agency"] == "GTFSDE") & (data_chuuchuu["routeType"] == "RE")

re_arverio_ranges = [(57006, 57347), (78900, 78984)]
re_arverio_mask = gtfsde_re_base & in_ranges(data_chuuchuu["routeNumber_int"], re_arverio_ranges)
data_chuuchuu.loc[re_arverio_mask, "operator"] = "Arverio"

re_db_stop_ids = ['8000042', '8000124', '8000131', '8000156', '8000189', '8000191', '8000218', '8000229', '8000236', '8000244', '8000264', '8000265', '8000275', '8000295', '8000323', '8000369', '8000373', '8000383', '8000423', '8000471', '8000599', '8000649', '8000668', '8000681', '8000736', '8001132', '8001366', '8001618', '8001707', '8001883', '8002021', '8002137', '8002342', '8002380', '8002632', '8002685', '8002883', '8002931', '8003101', '8003235', '8003726', '8003759', '8003932', '8004094', '8004095', '8004215', '8004219', '8004577', '8004658', '8005013', '8005077', '8005229', '8005494', '8005578', '8005592', '8005714', '8005736', '8006083', '8006137', '8006661', '8070097', '8700271', '8700439']
re_db_ranges = [(38761, 38770), (38784, 38799), (4280, 4299), (13324, 13334), (86381, 86391), (88824, 88873)]
re_db_single_numbers = {"38632", "38710", "38112"}

re_db_mask = (
    gtfsde_re_base
    & data_chuuchuu["deutscheBahnStopId"].isin(re_db_stop_ids)
    & (in_ranges(data_chuuchuu["routeNumber_int"], re_db_ranges) | data_chuuchuu["routeNumber_trim"].isin(re_db_single_numbers))
)
data_chuuchuu.loc[re_db_mask, "operator"] = "DB"
print(f"{re_arverio_mask.sum()} GTFSDE RE rows assigned 'Arverio', {re_db_mask.sum()} assigned 'DB'")

# RS
gtfsde_rs_base = (data_chuuchuu["agency"] == "GTFSDE") & (data_chuuchuu["routeType"] == "RS")
rs_db_mask = gtfsde_rs_base & in_ranges(data_chuuchuu["routeNumber_int"], [(32600, 32677), (57440, 57798)])
data_chuuchuu.loc[rs_db_mask, "operator"] = "DB"
print(f"{rs_db_mask.sum()} GTFSDE RS rows assigned 'DB'")

# U
gtfsde_u_base = (data_chuuchuu["agency"] == "GTFSDE") & (data_chuuchuu["routeType"] == "U")
u_db_mask = gtfsde_u_base & (data_chuuchuu["routeNumber_int"].between(28000, 28014) | (data_chuuchuu["routeNumber_int"] == 28023))
data_chuuchuu.loc[u_db_mask, "operator"] = "DB"
print(f"{u_db_mask.sum()} GTFSDE U rows assigned 'DB'")

print("GTFSDE EC/ECE/EN: intentionally left unassigned -- duplicates of DB's own reporting")

0 GTFSDE FEX rows assigned 'DB'
0 GTFSDE MEX rows assigned 'DB', 0 assigned 'Arverio'
0 GTFSDE NRB rows assigned 'DB'


0 GTFSDE RB rows assigned 'Arverio', 0 assigned 'DB'
0 GTFSDE RE rows assigned 'Arverio', 0 assigned 'DB'


0 GTFSDE RS rows assigned 'DB'
0 GTFSDE U rows assigned 'DB'
GTFSDE EC/ECE/EN: intentionally left unassigned -- duplicates of DB's own reporting


### OEBB agency: corrections from a later re-investigation

These come from the data provider re-examining OEBB's own EC/EN/IC reporting specifically, and **intentionally override** the earlier OEBB EC section above and the generic `EN` night-train assignment (in the night train services section) for these particular `(agency=OEBB, routeType, routeNumber)` combinations -- e.g. some `EN` route numbers previously assigned `HZ`/`MAV` by the generic table turn out to actually be `OEBB`/`PKP Intercity` when reported under the OEBB agency specifically.

Note: the source instructions' heading says "agency OEBB and SBB", but the actual condition only filters `agency = 'OEBB'` -- implemented exactly as the condition states, not the heading.

In [18]:
oebb_correction_db_ec_numbers = {"115", "1281", "190", "192", "194", "196", "198", "213", "80", "81", "94", "96", "98"}
oebb_correction_db_ic_numbers = {"406", "416"}
oebb_correction_db_mask = (
    (data_chuuchuu["agency"] == "OEBB")
    & (
        ((data_chuuchuu["routeType"] == "EC") & data_chuuchuu["routeNumber_trim"].isin(oebb_correction_db_ec_numbers))
        | ((data_chuuchuu["routeType"] == "IC") & data_chuuchuu["routeNumber_trim"].isin(oebb_correction_db_ic_numbers))
    )
)
data_chuuchuu.loc[oebb_correction_db_mask, "operator"] = "DB"

oebb_correction_oebb_ec_numbers = {
    "100", "102", "106", "114", "140", "141", "142", "143", "144", "145", "146", "147", "148", "149", "164",
    "202", "204", "206", "212", "214", "290", "337", "340", "341", "342", "343", "462", "463", "70", "71", "78", "79",
}
oebb_correction_oebb_en_numbers = {"40237", "40414", "40462", "40465", "40467", "414", "50237"}
oebb_correction_oebb_ic_numbers = {
    "1110", "1111", "1112", "1113", "1115", "1118", "1119", "1135", "1136", "1138", "1142", "1143", "1151", "1244", "1249",
    "350", "351", "354", "407", "460", "532", "533", "534", "535", "536", "537", "538", "540", "541", "542", "543", "544",
    "545", "546", "547", "548", "549", "558", "559", "640", "641", "642", "643", "644", "645", "646", "647", "648", "649",
    "651", "740", "741", "742", "743", "744", "745", "746", "747", "748", "749", "756", "759", "790", "791", "792", "793",
    "794", "795", "796", "797", "798", "799", "840", "841", "842", "843", "847", "848", "850", "890", "891", "896", "897",
    "898", "899",
}
oebb_correction_oebb_mask = (
    (data_chuuchuu["agency"] == "OEBB")
    & (
        ((data_chuuchuu["routeType"] == "EC") & data_chuuchuu["routeNumber_trim"].isin(oebb_correction_oebb_ec_numbers))
        | ((data_chuuchuu["routeType"] == "EN") & data_chuuchuu["routeNumber_trim"].isin(oebb_correction_oebb_en_numbers))
        | ((data_chuuchuu["routeType"] == "IC") & data_chuuchuu["routeNumber_trim"].isin(oebb_correction_oebb_ic_numbers))
    )
)
data_chuuchuu.loc[oebb_correction_oebb_mask, "operator"] = "OEBB"

oebb_correction_pkp_ec_numbers = {"101", "103", "107", "203", "205", "207"}
oebb_correction_pkp_en_numbers = {"40406", "40407", "40416", "40417"}
oebb_correction_pkp_ic_numbers = {"207", "417"}
oebb_correction_pkp_mask = (
    (data_chuuchuu["agency"] == "OEBB")
    & (
        ((data_chuuchuu["routeType"] == "EC") & data_chuuchuu["routeNumber_trim"].isin(oebb_correction_pkp_ec_numbers))
        | ((data_chuuchuu["routeType"] == "EN") & data_chuuchuu["routeNumber_trim"].isin(oebb_correction_pkp_en_numbers))
        | ((data_chuuchuu["routeType"] == "IC") & data_chuuchuu["routeNumber_trim"].isin(oebb_correction_pkp_ic_numbers))
    )
)
data_chuuchuu.loc[oebb_correction_pkp_mask, "operator"] = "PKP Intercity"

oebb_correction_sbb_ec_numbers = {"163", "191", "193", "195", "197", "199", "95", "97", "99"}
oebb_correction_sbb_mask = (
    (data_chuuchuu["agency"] == "OEBB") & (data_chuuchuu["routeType"] == "EC")
    & data_chuuchuu["routeNumber_trim"].isin(oebb_correction_sbb_ec_numbers)
)
data_chuuchuu.loc[oebb_correction_sbb_mask, "operator"] = "SBB"

print(f"OEBB corrections applied: {oebb_correction_db_mask.sum()} -> DB, {oebb_correction_oebb_mask.sum()} -> OEBB, {oebb_correction_pkp_mask.sum()} -> PKP Intercity, {oebb_correction_sbb_mask.sum()} -> SBB")

OEBB corrections applied: 0 -> DB, 0 -> OEBB, 0 -> PKP Intercity, 0 -> SBB


### Fallback: assume the local national operator where nothing else assigned one

For a handful of remaining unassigned combinations, the data provider suggests falling back to "whichever operator is national to the country the stop is in" -- applied only where `operator` is still null, so it never overrides anything assigned above.

In [19]:
# DB agency, EN routeType: fill in the leftover (e.g. the "no info" route numbers listed in the
# night train section) by country
db_en_fallback_map = {"Germany": "DB", "Austria": "OEBB", "Switzerland": "SBB", "Poland": "PKP Intercity", "Czech Republic": "CD"}
db_en_fallback_mask = (data_chuuchuu["agency"] == "DB") & (data_chuuchuu["routeType"] == "EN") & data_chuuchuu["operator"].isna()
data_chuuchuu.loc[db_en_fallback_mask, "operator"] = data_chuuchuu.loc[db_en_fallback_mask, "country"].map(db_en_fallback_map)
print(f"{db_en_fallback_mask.sum()} DB EN rows with no operator yet had a country-based fallback attempted")

# OEBB agency, EN/EC/IC/ECE routeTypes -- the source instructions' heading mentions "OEBB and SBB"
# but the actual condition only filters agency = 'OEBB'; implemented as the condition states
oebb_fallback_map = {
    "Germany": "DB", "Austria": "OEBB", "Switzerland": "SBB", "Poland": "PKP Intercity",
    "Czech Republic": "CD", "Italy": "Trenitalia",
}
oebb_fallback_mask = (
    (data_chuuchuu["agency"] == "OEBB") & data_chuuchuu["routeType"].isin(["EN", "EC", "IC", "ECE"])
    & data_chuuchuu["operator"].isna()
)
data_chuuchuu.loc[oebb_fallback_mask, "operator"] = data_chuuchuu.loc[oebb_fallback_mask, "country"].map(oebb_fallback_map)
print(f"{oebb_fallback_mask.sum()} OEBB EN/EC/IC/ECE rows with no operator yet had a country-based fallback attempted")

# IT agency, EC routeType: not covered by the earlier "easy operator cases" IT rule (which didn't
# include EC) -- always Trenitalia when the stop is in Italy or France
it_ec_fallback_mask = (
    (data_chuuchuu["agency"] == "IT") & (data_chuuchuu["routeType"] == "EC") & data_chuuchuu["operator"].isna()
    & data_chuuchuu["country"].isin(["Italy", "France"])
)
data_chuuchuu.loc[it_ec_fallback_mask, "operator"] = "Trenitalia"
print(f"{it_ec_fallback_mask.sum()} IT EC rows with no operator yet assigned 'Trenitalia' (Italy/France)")

0 DB EN rows with no operator yet had a country-based fallback attempted
0 OEBB EN/EC/IC/ECE rows with no operator yet had a country-based fallback attempted


0 IT EC rows with no operator yet assigned 'Trenitalia' (Italy/France)


In [20]:
data_chuuchuu["normalized_operator"] = data_chuuchuu["operator"]

db_regio_mask = data_chuuchuu["operator"].str.contains("DB Regio", case=False, na=False)
data_chuuchuu.loc[db_regio_mask, "normalized_operator"] = "DB Regio"
print(f"{db_regio_mask.sum()} rows with operator containing 'DB Regio' assigned 'DB Regio'")

0 rows with operator containing 'DB Regio' assigned 'DB Regio'


### Cleanup & summary

In [21]:
try:
    data_chuuchuu = data_chuuchuu.drop(columns=["routeType_norm", "agency_norm", "routeNumber_trim", "routeNumber_int"])
except KeyError:
    pass

print(f"{data_chuuchuu['normalized_operator'].isna().sum()} rows ({data_chuuchuu['normalized_operator'].isna().mean() * 100:.2f}%) still have no normalized_operator")
print()
data_chuuchuu["normalized_operator"].value_counts(dropna=False).head(30)

159979 rows (5.53%) still have no normalized_operator



normalized_operator
SNCF             2517370
NaN               159979
Eurostar          153926
Deutsche Bahn      38336
Trenitalia         24854
Name: count, dtype: int64

##  Verifications steps

Check the services for which operators are still not identified

In [22]:
op_to_check = "None"  # set to "None" (as a string) to check rows with no normalized_operator, or a real value like "OEBB"

if op_to_check == "None":
    operator_mask = data_chuuchuu["normalized_operator"].isna()
else:
    operator_mask = data_chuuchuu["normalized_operator"] == op_to_check

print(data_chuuchuu.loc[operator_mask, "country"].value_counts(dropna=False))
print()
print(data_chuuchuu.loc[operator_mask, "routeType"].value_counts(dropna=False))
print()
print(data_chuuchuu.loc[operator_mask, "journey_type"].value_counts(dropna=False))
print()
print(data_chuuchuu.loc[operator_mask, "agency"].value_counts(dropna=False))

country
France         159448
Germany           451
Switzerland        79
Italy               1
Name: count, dtype: int64

routeType
CAR TER      119320
TRAMTRAIN     40659
Name: count, dtype: int64

journey_type
domestic         158463
international      1516
Name: count, dtype: int64

agency
FR    159979
Name: count, dtype: int64


In [23]:
#  check op_to_check == "None":
operator_mask = data_chuuchuu["normalized_operator"].isna()

for route_type in data_chuuchuu.loc[operator_mask,"routeType"].unique():
    route_type_mask = data_chuuchuu["routeType"] == route_type
    print(f"Route type to check: {route_type}")
    print()
    print(data_chuuchuu.loc[operator_mask & route_type_mask, "journey_type"].value_counts(dropna=False))
    print()
    print(data_chuuchuu.loc[operator_mask & route_type_mask, "country"].value_counts(dropna=False))
    print()
    print(data_chuuchuu.loc[operator_mask & route_type_mask, "agency"].value_counts(dropna=False))
    print("#################################################")
    print("#################################################")
    print()

Route type to check: CAR TER

journey_type
domestic         117924
international      1396
Name: count, dtype: int64

country
France         118849
Germany           391
Switzerland        79
Italy               1
Name: count, dtype: int64

agency
FR    119320
Name: count, dtype: int64
#################################################
#################################################

Route type to check: TRAMTRAIN

journey_type
domestic         40539
international      120
Name: count, dtype: int64

country
France     40599
Germany       60
Name: count, dtype: int64

agency
FR    40659
Name: count, dtype: int64
#################################################
#################################################



In [24]:
operator_mask = data_chuuchuu["normalized_operator"].isna()
route_type_mask = data_chuuchuu["routeType"] == "RB"
agency_mask = data_chuuchuu["agency"] == "SBB"

print(data_chuuchuu.loc[operator_mask & route_type_mask & agency_mask, "journey_type"].value_counts(dropna=False))
print()
print(data_chuuchuu.loc[operator_mask & route_type_mask & agency_mask, "country"].value_counts(dropna=False))

Series([], Name: count, dtype: int64)

Series([], Name: count, dtype: int64)


What operators are available for data 2025 ?

In [25]:
agencies_after_2025 = ["PL", "HU", "GTFSDE", "DK", "SBB", "OEBB", "RENFE"]

before_2025 = ~data_chuuchuu["agency"].isin(agencies_after_2025)
missing_normalized_operator = data_chuuchuu.loc[before_2025, "normalized_operator"].isna()
print(f"{missing_normalized_operator.sum()} rows ({missing_normalized_operator.mean() * 100:.2f}%) still have no normalized_operator (excluding {agencies_after_2025})")
print()
data_chuuchuu.loc[before_2025, "normalized_operator"].value_counts(dropna=False).head(30)

159979 rows (5.53%) still have no normalized_operator (excluding ['PL', 'HU', 'GTFSDE', 'DK', 'SBB', 'OEBB', 'RENFE'])



normalized_operator
SNCF             2517370
NaN               159979
Eurostar          153926
Deutsche Bahn      38336
Trenitalia         24854
Name: count, dtype: int64

Check which operators are really present (mostly domestic market) for the 2025 data:

In [26]:
op_to_check = "DSB"  # set to "None" (as a string) to check rows with no normalized_operator, or a real value like "OEBB"

if op_to_check == "None":
    operator_mask = data_chuuchuu["normalized_operator"].isna()
else:
    operator_mask = data_chuuchuu["normalized_operator"] == op_to_check

print(data_chuuchuu.loc[before_2025 & operator_mask, "country"].value_counts(dropna=False))
print()
print(data_chuuchuu.loc[before_2025 & operator_mask, "routeType"].value_counts(dropna=False))
print()
print(data_chuuchuu.loc[before_2025 & operator_mask, "journey_type"].value_counts(dropna=False))
print()
print(data_chuuchuu.loc[before_2025 & operator_mask, "agency"].value_counts(dropna=False))

Series([], Name: count, dtype: int64)

Series([], Name: count, dtype: int64)

Series([], Name: count, dtype: int64)

Series([], Name: count, dtype: int64)


In [27]:
op_to_check = "None"  # set to "None" (as a string) to check rows with no normalized_operator, or a real value like "OEBB"

route_type_to_check = "EN"  # this is a routeType value (e.g. "Sprinter", "ICE"), not a journey_type value

if op_to_check == "None":
    operator_mask = data_chuuchuu["normalized_operator"].isna()
else:
    operator_mask = data_chuuchuu["normalized_operator"] == op_to_check

route_type_mask = data_chuuchuu["routeType"] == route_type_to_check

print(data_chuuchuu.loc[before_2025 & operator_mask & route_type_mask, "country"].value_counts(dropna=False))
print()
print(data_chuuchuu.loc[before_2025 & operator_mask & route_type_mask, "journey_type"].value_counts(dropna=False))
print()
print(data_chuuchuu.loc[before_2025 & operator_mask & route_type_mask, "agency"].value_counts(dropna=False))
print()
print(data_chuuchuu.loc[before_2025 & operator_mask & route_type_mask, "routeNumber"].value_counts(dropna=False))

Series([], Name: count, dtype: int64)

Series([], Name: count, dtype: int64)

Series([], Name: count, dtype: int64)

Series([], Name: count, dtype: int64)


### Deduplicating trains reported by multiple agencies

Per the data provider: the same physical train can be collected by more than one agency (e.g. a Nightjet through Venice reported by both OEBB as `NJ` and IT as `EN`). A row is uniquely identified by (`routeNumber`, `deutscheBahnStopId`, `plannedArrival`, `plannedDeparture`) -- only one train can have a given number and planned arrival/departure at a given station. Within each duplicate group we keep a single row, preferring (in order):

1. the row where `arrivalCancelled` is true (kept as the most informative one),
2. otherwise the row with the highest `arrivalDelay`,
3. a stable tiebreaker on the original row order (mirrors the SQL's `rowid ASC`).

This mirrors the provider's `ROW_NUMBER() OVER (PARTITION BY ... ORDER BY ...) = 1` query: a global sort by the same ORDER BY keys, followed by `duplicated(keep="first")` on the partition columns, picks exactly the same row per group.

**Caveat handled below:** pandas' `duplicated()` (like SQL's `PARTITION BY`) treats a shared `NaN` as a match, so two unrelated rows that both happen to be missing `plannedArrival`/`plannedDeparture` would otherwise collapse into the same "group" purely because they share a missing value, not because they're the same train. Rows missing either planned time are therefore excluded from dedup consideration entirely and kept as-is, rather than risking an incorrect merge.

In [28]:
dedup_keys = ["routeNumber", "deutscheBahnStopId", "plannedArrival", "plannedDeparture"]

# only rows where both planned times are known form a reliable dedup key -- pandas' duplicated()
# (like SQL's PARTITION BY) treats a shared NaN as a match, but a missing planned time isn't
# evidence of being the same physical train, so those rows are excluded from dedup consideration
# entirely and kept as-is
has_full_planned_times = data_chuuchuu["plannedArrival"].notna() & data_chuuchuu["plannedDeparture"].notna()
print(f"{(~has_full_planned_times).sum()} rows are missing a planned time and skip deduplication entirely")

# priority 0 = cancelled (kept first, the "most informative" row), 1 = everything else -- mirrors the SQL CASE.
# spelling out the case variants directly (rather than .astype(str).str.lower()) avoids materializing
# a full string-object copy of all 15M rows just to fold case, which was blowing up on memory.
cancelled_priority = (~data_chuuchuu["arrivalCancelled"].isin(["t", "T", "1", "true", "True", "TRUE"])).astype(int)

# an explicit row-order column so the final tiebreaker matches the SQL's `rowid ASC` on the original row order
row_order = np.arange(len(data_chuuchuu))

dedup_sort_key = pd.DataFrame({
    "cancelled_priority": cancelled_priority.to_numpy(),
    "arrivalDelay": data_chuuchuu["arrivalDelay"].to_numpy(),
    "row_order": row_order,
}, index=data_chuuchuu.index)

# a global sort by the same ORDER BY keys as the SQL window function, then duplicated(keep="first")
# on the partition columns picks the same single row per (routeNumber, deutscheBahnStopId,
# plannedArrival, plannedDeparture) group that ROW_NUMBER() OVER (PARTITION BY ... ORDER BY ...) = 1 would
sorted_index = dedup_sort_key.loc[has_full_planned_times].sort_values(
    by=["cancelled_priority", "arrivalDelay", "row_order"],
    ascending=[True, False, True],
    na_position="last",
).index

is_duplicate = data_chuuchuu.loc[sorted_index, dedup_keys].duplicated(keep="first")
rows_to_drop = is_duplicate[is_duplicate].index

print(f"{len(rows_to_drop)} duplicate rows removed ({len(rows_to_drop) / len(data_chuuchuu) * 100:.2f}%)")

804300 rows are missing a planned time and skip deduplication entirely


13 duplicate rows removed (0.00%)


In [29]:
data_chuuchuu = data_chuuchuu.drop(index=rows_to_drop)
# avoid reset_index(drop=True) here: on a frame this wide, it triggers an internal deep copy that
# consolidates every object-dtype column into one contiguous block via np.vstack, which needs a single
# multi-GiB allocation and is what blew up on memory. Reassigning .index directly is metadata-only --
# no data copy, no consolidation -- and produces the same clean 0..N-1 index.
data_chuuchuu.index = np.arange(len(data_chuuchuu))
data_chuuchuu.shape

(2894452, 36)

## Exporting data

In [30]:
export_data = input("Export intermediate data to parquet? (y/n): ")

if export_data.lower() == "y":
    os.makedirs(intermediate_outputs_dir, exist_ok=True)

    data_chuuchuu.to_parquet(f"{intermediate_outputs_dir}/data_chuuchuu_{data_selection}_operators.parquet")